# Monitoring + teardown

A synchronous endpoint's health is latency, error rate, and drift in what
callers send. Locally we inspect the serving container and the store, then
tear the stack down.

In [ ]:
import docker

client = docker.from_env()
[
    c.name
    for c in client.containers.list()
    if "pricing" in c.name or "california" in c.name
]

In [ ]:
%%bash
docker logs --tail 30 $(docker ps -q --filter name=sagemaker) 2>/dev/null | tail -15

## Endpoint-side sanity

In [ ]:
# distribution of served predictions vs training distribution is the
# drift check that matters for a request-time endpoint — see
# evaluation.ipynb for residual analysis; production adds invoke
# latency metrics on top.
served_note = "endpoint-side drift lives in the invoke stream"
served_note

## Teardown

In [ ]:
from sagemaker.local import LocalSession

try:
    LocalSession().sagemaker_client.delete_endpoint(
        EndpointName="california-pricing"
    )
except Exception as e:
    print("endpoint cleanup:", e)

In [ ]:
%%bash
make down